In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install transformers datasets accelerate

In [3]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments
from datasets import Dataset

In [4]:

corpus_path = "/content/drive/MyDrive/RTRP/data/corpus.txt"

with open(corpus_path, "r", encoding="utf-8") as f:
    text_data = f.read()

print("Corpus loaded successfully")
print("Total characters:", len(text_data))

Corpus loaded successfully
Total characters: 4007599


In [5]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: Tesla T4


In [6]:
!pip install transformers datasets accelerate

In [7]:
!pip install datasets

In [8]:
from datasets import load_dataset

In [9]:
data_path = "/content/drive/MyDrive/RTRP/data/corpus.txt"

dataset = load_dataset("text", data_files=data_path)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 30764
    })
})


In [10]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

print("Model & Tokenizer Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model & Tokenizer Loaded


In [17]:
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # IMPORTANT — labels add kar rahe hain
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print("Tokenization with labels complete")

Map:   0%|          | 0/30764 [00:00<?, ? examples/s]

Tokenization with labels complete


In [12]:
!pip install -U transformers accelerate

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/RTRP/models",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    logging_dir="/content/drive/MyDrive/RTRP/models/logs",
    report_to="none",
    fp16=True
)

print("Training arguments ready")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments ready


In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

print("Trainer ready")

Trainer ready


In [19]:
print(tokenized_dataset["train"].column_names)

['input_ids', 'attention_mask', 'labels']


In [20]:
model.train()
print("Model set to train mode")

Model set to train mode


In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

print("Fresh trainer created")

Fresh trainer created


In [22]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,1.425544
200,0.558849
300,0.470718
400,0.480285
500,0.476507
600,0.426611
700,0.432212
800,0.434887
900,0.445303
1000,0.405497


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7691, training_loss=0.36494241351657225, metrics={'train_runtime': 971.7827, 'train_samples_per_second': 31.657, 'train_steps_per_second': 7.914, 'total_flos': 2009597018112000.0, 'train_loss': 0.36494241351657225, 'epoch': 1.0})

In [26]:
print(os.listdir("drive"))

['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']


In [27]:
print(os.listdir("drive/MyDrive"))

['IMG-20250303-WA0005.jpg', '24BD1A6632 (2).jpg', '24BD1A6632 (1).jpg', '24BD1A6632.jpg', 'Classroom', 'photoconvertinto20kb_com20240926_214655.jpg', 'IMG-20241220-WA0012 (2).jpg', 'IMG-20241220-WA0012 (1).jpg', 'IMG-20241220-WA0012.jpg', 'keshav.pdf', 'DSA SHEET.gsheet', "Copy of DSA by Shradha Ma'am.gsheet", '24BD1A6632.pdf', 'invoice.pdf', 'Programming In Java_page-0001.jpg', 'IMG-20250610-WA0003.jpg', 'AI Unit-1.gdoc', 'BACKUP', 'Copy of JAVA-1 (2).pdf', 'Copy of JAVA-1 (1).pdf', 'Copy of JAVA-1.pdf', 'IMG-20251122-WA0000.jpg', 'Document from Keshav Rathi (1).pdf', 'Document from Keshav Rathi.pdf', 'Keshav_Rathi_Resume.pdf', 'image_ef8375c5-b5dd-41e9-9b4d-b6d28ae4ff9520251211_104657.jpg', 'image_33319fe2-2968-47ea-adb3-6499e54ab40d20251211_104701.jpg', 'image_7f9b91c6-2817-49df-8dd6-45436d0efe5320251211_104705.jpg', 'image_63b6a431-f74e-4dae-ab8e-2975f8cbd5c920251211_104721.jpg', 'Copy of DBMS Unit 5 Tesseract.pdf', 'Untitled presentation.gslides', 'WhatsApp Image 2026-02-18 at 11.

In [30]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["train"]   # temporary
)

print("Trainer recreated with eval dataset")

Trainer recreated with eval dataset


In [31]:
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 0.27135542035102844, 'eval_model_preparation_time': 0.0025, 'eval_runtime': 162.9634, 'eval_samples_per_second': 188.779, 'eval_steps_per_second': 23.6}
